# Phase 2 — LoRA-QAT (INT8 & INT4)

**Goal:** measure the cheapest QAT variant. Quantize the base model first, then attach small LoRA adapters (rank 16, ~1–2% of total parameters) to recover quality. Base weights stay frozen and quantized; only the adapters train.

**Why this matters:** if LoRA-QAT lands close to full Scheduled QAT, the cost-quality tradeoff favours it for production deployments where you can't afford a full QAT run per model variant.

**Inputs:** `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs:**
- `models/checkpoints/lora_qat_int{4,8}/final_adapter/` (PEFT adapter dirs)
- `results/lora_qat_int{4,8}/training_results.json`

**Estimated time on T4:** ~2.5 h total (fewer trainable params → fewer optimizer FLOPs per step).

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"
BASELINE_DIR = "/kaggle/input/sqat-baseline/results/baseline"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), "FP32 logits not found — add sqat-baseline as input"

In [ ]:
# peft is needed for LoRA adapters; viz for plots.
!pip install -q -e ".[viz]" peft
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

## 2. Kaggle config overrides

LoRA-QAT uses `epochs=2` and a higher LR (2e-4) in the published config because only the adapters train. We keep those choices and just shrink seq_length / batch / accum to fit Kaggle.

In [ ]:
import yaml

MODEL_CACHE = f"{WORKING_DIR}/models/base"
KAGGLE_CFG  = Path(REPO_DIR) / "configs_kaggle" / "lora_qat"
KAGGLE_CFG.mkdir(parents=True, exist_ok=True)

def patch_lora_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = 1               # 1 epoch is enough at LoRA's higher LR
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 50
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{WORKING_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{WORKING_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

lora8_cfg = patch_lora_config("configs/lora_qat/lora_qat_int8.yaml", KAGGLE_CFG / "lora_qat_int8.yaml")
lora4_cfg = patch_lora_config("configs/lora_qat/lora_qat_int4.yaml", KAGGLE_CFG / "lora_qat_int4.yaml")

print("INT8 config:", lora8_cfg)
print("INT4 config:", lora4_cfg)

## 3. LoRA-QAT — INT8

Frozen quantized base + trainable rank-16 LoRA adapters on q/k/v/o projections.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_int8 = run_experiment(
    config_path=str(lora8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLoRA-QAT INT8:")
for k, v in results_int8.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. LoRA-QAT — INT4

Same setup, INT4 base. The adapters compensate for the larger quantization error.

In [ ]:
results_int4 = run_experiment(
    config_path=str(lora4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLoRA-QAT INT4:")
for k, v in results_int4.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Adapter parameter count

Quick sanity check on how many parameters were actually trained. For SmolLM2-1.7B with rank 16 on q/k/v/o, expect on the order of 4M trainable params (~0.25% of total). The exact number depends on hidden_size and num_attention_heads.

In [ ]:
# Reload the INT4 adapter and report trainable param count.
from peft import PeftModel
from transformers import AutoModelForCausalLM

adapter_dir = Path(WORKING_DIR) / "models" / "checkpoints" / "lora_qat_int4" / "final_adapter"
if adapter_dir.exists():
    base = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
    )
    peft_model = PeftModel.from_pretrained(base, str(adapter_dir))
    total = sum(p.numel() for p in peft_model.parameters())
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"Total params      : {total:,}")
    print(f"Trainable (LoRA)  : {trainable:,}")
    print(f"Trainable fraction: {trainable / total * 100:.3f}%")
    del peft_model, base
    torch.cuda.empty_cache()
else:
    print(f"Adapter dir not found: {adapter_dir}")

## 6. Comparison table

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32",     "bits": 32, "perplexity": fp32["perplexity"], "kl_divergence": 0.0},
    {"method": "LoRA-QAT", "bits": 8,  "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence")},
    {"method": "LoRA-QAT", "bits": 4,  "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence")},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(WORKING_DIR) / "results" / "lora_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w") as f:
    json.dump({"int8": results_int8, "int4": results_int4, "fp32": fp32},
              f, indent=2, default=str)
print(f"Wrote {summary_path}")

## Next steps

Save outputs as `sqat-lora-qat`. Proceed to `06_export_gguf.ipynb` for edge deployment formats.